In [ ]:
# ==========================================
# 1. INSTALL LIBRARIES
# ==========================================
!pip install transformers torch scikit-learn matplotlib seaborn

# ==========================================_
# 2. IMPORT LIBRARIES
# ==========================================
import pandas as pd
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from torch.optim import AdamW

# ==========================================
# 3. UPLOAD DATASET
# ==========================================
from google.colab import files
uploaded = files.upload()

df = pd.read_csv(list(uploaded.keys())[0])

# ==========================================
# 4. DATA PREPROCESSING (MANDATORY)
# ==========================================
# For IMDB dataset
df = df[['review', 'sentiment']]

# Handle missing values
df = df.dropna()

# Convert labels to numeric
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Rename columns
df = df[['review', 'label']]
df.rename(columns={'review': 'text'}, inplace=True)

# Lowercase
df['text'] = df['text'].str.lower()

print(df.head())

# ==========================================
# 5. DATA SPLITTING (TRAIN/VAL/TEST)
# ==========================================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'], df['label'], test_size=0.3, random_state=42)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42)

# ==========================================
# 6. TOKENIZATION
# ==========================================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(train_texts.tolist(), truncation=True, padding=True)
val_encodings = tokenizer(val_texts.tolist(), truncation=True, padding=True)
test_encodings = tokenizer(test_texts.tolist(), truncation=True, padding=True)

# ==========================================
# 7. DATASET CLASS
# ==========================================
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)
test_dataset = Dataset(test_encodings, test_labels)

# ==========================================
# 8. METRICS FUNCTION (MANDATORY)
# ==========================================
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# ==========================================
# FUNCTION TO TRAIN MODEL
# ==========================================
def train_model(model, train_dataset, val_dataset):
    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=2,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        eval_strategy="epoch", # Corrected evaluation_strategy to eval_strategy
        save_strategy="epoch",
        logging_dir="./logs"
    )

    optimizer = AdamW(model.parameters(), lr=2e-5)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        optimizers=(optimizer, None)
    )

    trainer.train()
    return trainer

# ==========================================
# 9. EXPERIMENT 1 - FULL FINE-TUNING
# ==========================================
print("Running Full Fine-Tuning...")
model1 = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
trainer1 = train_model(model1, train_dataset, val_dataset)
pred1 = trainer1.predict(test_dataset)
print("Full Fine-Tuning Metrics:", pred1.metrics)

# ==========================================
# 10. EXPERIMENT 2 - FREEZE BERT
# ==========================================
print("Running Frozen BERT...")
model2 = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

for param in model2.bert.parameters():
    param.requires_grad = False

trainer2 = train_model(model2, train_dataset, val_dataset)
pred2 = trainer2.predict(test_dataset)
print("Frozen BERT Metrics:", pred2.metrics)

# ==========================================
# 11. EXPERIMENT 3 - LAST 2 LAYERS
# ==========================================
print("Running Last 2 Layers Fine-Tuning...")
model3 = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

for name, param in model3.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainer3 = train_model(model3, train_dataset, val_dataset)
pred3 = trainer3.predict(test_dataset)
print("Last 2 Layers Metrics:", pred3.metrics)

# ==========================================
# 12. CONFUSION MATRIX (MANDATORY)
# ==========================================
y_true = test_labels.tolist()
y_pred = pred1.predictions.argmax(axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# ==========================================
# 13. COMPARISON OUTPUT
# ==========================================
print("\n===== FINAL COMPARISON =====")
print("Full Fine-Tuning:", pred1.metrics)
print("Frozen BERT:", pred2.metrics)
print("Last 2 Layers:", pred3.metrics)


Saving IMDB Dataset.csv to IMDB Dataset (3).csv
                                                text  label
0  one of the other reviewers has mentioned that ...      1
1  a wonderful little production. <br /><br />the...      1
2  i thought this was a wonderful way to spend ti...      1
3  basically there's a family where a little boy ...      0
4  petter mattei's "love in the time of money" is...      1
Running Full Fine-Tuning...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

POS Tagging vs Chunking
Feature	POS Tagging	Chunking
Level	Word-level	Phrase-level
Example	Noun, Verb	Noun Phrase (NP)
Difficulty	Easy	Medium
Output	Grammar tags	Grouped phrases
🔹 Report Section
Challenges:
Handling subword tokens
Label alignment (-100 issue)
Training time
Observations:
DistilBERT works well for sequence labeling
Chunking is slightly harder than POS tagging